# Pipeline: Instalar e Configurar VS Code + Python + Extensoes Essenciais

Este notebook apresenta um passo a passo reproduzivel para Windows, macOS e Linux com comandos prontos para uso.

## Como usar
1. Execute as celulas em ordem.
2. Se algum comando falhar, leia a secao de troubleshooting.
3. Sempre prefira o terminal integrado do VS Code.

## 1) Detectar SO e Pre-requisitos

Esta etapa detecta o sistema operacional e verifica se os gerenciadores e ferramentas basicas estao disponiveis:
- Windows: `winget`
- macOS: `brew`
- Linux: `apt`
- Ferramentas: `code`, `python`, `pip`

In [ ]:
import platform
import shutil

so = platform.system()
shell = "PowerShell" if so == "Windows" else "Bash/Zsh"

print(f"SO detectado: {so}")
print(f"Shell recomendado: {shell}")

for ferramenta in ["winget", "brew", "apt", "code", "python", "python3", "py", "pip", "pip3"]:
    caminho = shutil.which(ferramenta)
    print(f"{ferramenta:8} -> {'OK: ' + caminho if caminho else 'nao encontrado'}")

## 2) Instalar Visual Studio Code via CLI

### Windows (PowerShell)
```powershell
winget install --id Microsoft.VisualStudioCode -e --source winget
```

### macOS (Terminal)
```bash
brew install --cask visual-studio-code
```

### Ubuntu/Debian (Terminal)
```bash
sudo apt update
sudo apt install -y code
```

Validacao:
```bash
code --version
```

In [ ]:
import shutil
import subprocess

if shutil.which("code"):
    resultado = subprocess.run(["code", "--version"], capture_output=True, text=True, check=False)
    print((resultado.stdout or resultado.stderr).strip())
else:
    print("Comando `code` nao encontrado. Instale o VS Code e habilite integracao no PATH.")

## 3) Instalar Python e Validar `pip`

### Windows (PowerShell)
```powershell
winget install --id Python.Python.3.12 -e
```

### macOS (Terminal)
```bash
brew install python
```

### Ubuntu/Debian (Terminal)
```bash
sudo apt update
sudo apt install -y python3 python3-pip python3-venv
```

Validacao:
```bash
python --version
python3 --version
py --version
pip --version
pip3 --version
```

In [ ]:
import shutil
import subprocess

comandos = [
    ["python", "--version"],
    ["python3", "--version"],
    ["py", "--version"],
    ["pip", "--version"],
    ["pip3", "--version"],
]

for cmd in comandos:
    if shutil.which(cmd[0]):
        r = subprocess.run(cmd, capture_output=True, text=True, check=False)
        saida = (r.stdout or r.stderr).strip()
        print(f"{' '.join(cmd):18} -> {saida}")
    else:
        print(f"{' '.join(cmd):18} -> comando nao encontrado")

## 4) Criar Pasta do Projeto e Ambiente Virtual

### PowerShell (Windows)
```powershell
mkdir dev_workspace
cd dev_workspace
python -m venv .venv
.\.venv\Scripts\Activate.ps1
```

### Bash/Zsh (macOS/Linux)
```bash
mkdir -p dev_workspace
cd dev_workspace
python3 -m venv .venv
source .venv/bin/activate
```

Resultado esperado: o prompt mostra `(.venv)` ativo.

In [ ]:
import os
import platform
from pathlib import Path

pasta = Path.cwd() / "dev_workspace"
pasta.mkdir(exist_ok=True)
os.chdir(pasta)

print(f"Pasta atual: {pasta}")
if platform.system() == "Windows":
    print("Execute no terminal: python -m venv .venv")
    print("Ative com: .\\.venv\\Scripts\\Activate.ps1")
else:
    print("Execute no terminal: python3 -m venv .venv")
    print("Ative com: source .venv/bin/activate")

## 5) Instalar Pacotes Essenciais de Desenvolvimento Python

Com o ambiente virtual ativo:

```bash
python -m pip install --upgrade pip
pip install ipykernel pytest black flake8 pylint mypy
pip freeze > requirements.txt
```

In [ ]:
from pathlib import Path

req = Path("requirements.txt")
print("Execute no terminal com .venv ativa:\n")
print("python -m pip install --upgrade pip")
print("pip install ipykernel pytest black flake8 pylint mypy")
print("pip freeze > requirements.txt")

if req.exists():
    print("\nrequirements.txt encontrado. Primeiras linhas:")
    print("\n".join(req.read_text(encoding="utf-8").splitlines()[:10]))
else:
    print("\nrequirements.txt ainda nao foi criado.")

## 6) Instalar Extensoes do VS Code com `code --install-extension`

Extensoes recomendadas:
- `ms-python.python`
- `ms-python.vscode-pylance`
- `ms-toolsai.jupyter`
- `ms-python.debugpy`
- `ms-python.black-formatter`
- `ms-python.isort`
- `ms-python.flake8`

Comandos:
```bash
code --install-extension ms-python.python
code --install-extension ms-python.vscode-pylance
code --install-extension ms-toolsai.jupyter
code --install-extension ms-python.debugpy
code --install-extension ms-python.black-formatter
code --install-extension ms-python.isort
code --install-extension ms-python.flake8
code --list-extensions
```

In [ ]:
import shutil
import subprocess

extensoes = [
    "ms-python.python",
    "ms-python.vscode-pylance",
    "ms-toolsai.jupyter",
    "ms-python.debugpy",
    "ms-python.black-formatter",
    "ms-python.isort",
    "ms-python.flake8",
]

if not shutil.which("code"):
    print("Comando `code` nao encontrado. Habilite o CLI do VS Code no PATH.")
else:
    for ext in extensoes:
        print(f"Instalando {ext}...")
        subprocess.run(["code", "--install-extension", ext], check=False)

    print("\nExtensoes instaladas:")
    subprocess.run(["code", "--list-extensions"], check=False)

## 7) Criar Configuracoes de Workspace Python no VS Code

Esta etapa gera o arquivo `.vscode/settings.json` com:
- Interpretador do `.venv`
- Formatacao com Black
- Descoberta de testes com pytest
- Configuracoes de notebook e terminal

In [ ]:
import json
import platform
from pathlib import Path

vscode_dir = Path(".vscode")
vscode_dir.mkdir(exist_ok=True)
settings_file = vscode_dir / "settings.json"

interpretador = "${workspaceFolder}/.venv/Scripts/python.exe"
if platform.system() != "Windows":
    interpretador = "${workspaceFolder}/.venv/bin/python"

settings = {
    "python.defaultInterpreterPath": interpretador,
    "python.terminal.activateEnvironment": True,
    "python.testing.pytestEnabled": True,
    "python.testing.unittestEnabled": False,
    "python.testing.pytestArgs": ["tests"],
    "python.linting.enabled": True,
    "editor.formatOnSave": True,
    "python.analysis.typeCheckingMode": "basic",
    "[python]": {
        "editor.defaultFormatter": "ms-python.black-formatter"
    },
    "jupyter.askForKernelRestart": False,
    "notebook.formatOnSave.enabled": True
}

settings_file.write_text(json.dumps(settings, indent=2), encoding="utf-8")
print(f"Arquivo criado: {settings_file}")
print(settings_file.read_text(encoding="utf-8"))

## 8) Rodar Script de Verificacao e Testes Unitarios no VS Code

Esta etapa cria:
- `hello_setup.py` para validar Python e imports
- `tests/test_setup.py` com um teste basico

Depois executa:
```bash
python hello_setup.py
pytest -q
```

In [ ]:
from pathlib import Path
import subprocess
import sys

hello = Path("hello_setup.py")
tests_dir = Path("tests")
tests_dir.mkdir(exist_ok=True)
test_file = tests_dir / "test_setup.py"

hello.write_text(
    """import sys\n\nmods = [\"pytest\", \"black\", \"flake8\", \"pylint\", \"mypy\"]\nprint(f\"Python: {sys.version}\")\n\nfor m in mods:\n    try:\n        __import__(m)\n        print(f\"[OK] {m}\")\n    except Exception as exc:\n        print(f\"[FALHA] {m}: {exc}\")\n""",
    encoding="utf-8",
)

test_file.write_text(
    """def test_soma_basica():\n    assert 2 + 2 == 4\n""",
    encoding="utf-8",
)

print(f"Criado: {hello}")
print(f"Criado: {test_file}")

print("\nExecutando validacao...\n")
subprocess.run([sys.executable, str(hello)], check=False)

print("\nExecutando pytest...\n")
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=False)

## Troubleshooting

1. `python` nao encontrado:
   - Feche e abra o terminal apos instalar.
   - No Windows, reinstale marcando "Add Python to PATH".
2. PowerShell nao ativa o `.venv`:
   - Execute `Set-ExecutionPolicy -Scope CurrentUser RemoteSigned` e reabra o terminal.
3. Interpretador errado no VS Code:
   - `Ctrl+Shift+P` -> `Python: Select Interpreter` -> selecione `.venv`.
4. `code` nao encontrado:
   - Reinstale o VS Code com integracao no PATH.

## Fluxo Diario Rapido

1. Abra a pasta do projeto no VS Code.
2. Ative o `.venv` no terminal.
3. Rode scripts/notebooks.
4. Formate ao salvar com Black.
5. Rode `pytest -q` antes de commit.